<a href="https://colab.research.google.com/github/harshith-murali/BIS-Lab/blob/main/Cuckoo_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


def cuckoo_feature_selection(X, y, n_nests=15, iterations=30,
                             pa=0.25, penalty=0.05, seed=42):
    rng = np.random.default_rng(seed)
    n_features = X.shape[1]

    # Test data is kept separate from feature selection.
    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev, y_dev, test_size=0.25,
        stratify=y_dev, random_state=seed
    )

    cache = {}

    def evaluate(position):
        # Convert continuous position into a binary feature mask.
        probabilities = 1 / (1 + np.exp(-position))
        mask = rng.random(n_features) < probabilities

        if not mask.any():
            mask[rng.integers(n_features)] = True

        key = tuple(mask)

        if key not in cache:
            model = DecisionTreeClassifier(
                max_depth=5, random_state=seed
            )
            model.fit(X_train[:, mask], y_train)
            predictions = model.predict(X_val[:, mask])
            accuracy = accuracy_score(y_val, predictions)

            cost = (
                1 - accuracy
                + penalty * mask.sum() / n_features
            )
            cache[key] = (cost, accuracy)

        cost, accuracy = cache[key]
        return cost, accuracy, mask

    def levy_flight():
        beta = 1.5
        sigma = (
            math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
            / (
                math.gamma((1 + beta) / 2)
                * beta * 2 ** ((beta - 1) / 2)
            )
        ) ** (1 / beta)

        u = rng.normal(0, sigma, n_features)
        v = rng.normal(0, 1, n_features)
        return u / (np.abs(v) + 1e-10) ** (1 / beta)

    nests = rng.uniform(-2, 2, (n_nests, n_features))
    results = [evaluate(position) for position in nests]

    best = min(results, key=lambda result: result[0])
    best_mask = best[2].copy()
    best_cost, best_accuracy = best[:2]

    for _ in range(iterations):
        for i in range(n_nests):
            candidate = np.clip(
                nests[i] + 0.5 * levy_flight(), -6, 6
            )
            result = evaluate(candidate)

            j = rng.integers(n_nests)
            if result[0] < results[j][0]:
                nests[j] = candidate
                results[j] = result

        # Replace a fraction of the worst nests.
        count = int(pa * n_nests)
        worst = sorted(
            range(n_nests),
            key=lambda j: results[j][0],
            reverse=True
        )[:count]

        for j in worst:
            nests[j] = rng.uniform(-2, 2, n_features)
            results[j] = evaluate(nests[j])

        # Preserve the best solution found across all iterations.
        current = min(results, key=lambda result: result[0])
        if current[0] < best_cost:
            best_cost, best_accuracy = current[:2]
            best_mask = current[2].copy()

    model = DecisionTreeClassifier(max_depth=5, random_state=seed)
    model.fit(X_dev[:, best_mask], y_dev)
    test_accuracy = accuracy_score(
        y_test, model.predict(X_test[:, best_mask])
    )

    return np.flatnonzero(best_mask), best_accuracy, test_accuracy


# Synthetic demonstration data.
# Replace X with numeric network features and y with intrusion labels.
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    random_state=42
)

selected, validation_accuracy, test_accuracy = (
    cuckoo_feature_selection(X, y)
)

print("Selected feature indices:", selected.tolist())
print("Number of selected features:", len(selected))
print(f"Validation accuracy: {validation_accuracy:.2%}")
print(f"Test accuracy: {test_accuracy:.2%}")

Selected feature indices: [0, 1, 2, 3, 5, 6, 13, 14, 16, 17, 18]
Number of selected features: 11
Validation accuracy: 83.50%
Test accuracy: 78.50%
